# 07 - Use case: bispecific cross-reactivity screen

**Scenario.** A bispecific / TCR-mimic therapeutic is designed against a target
peptide. Off-target risk comes from *self* peptides that are a single residue away
yet still bind the same TCR - exactly an activity cliff in reverse. Here we:

1. take a candidate **target peptide**,
2. enumerate all **single-residue neighbours** of it,
3. **score** them with a trained model,
4. **flag** neighbours predicted to bind (cross-reactivity risk),
5. export hits to **NetTCR-2.0** format and note **ANARCI/IMGT** CDR3 extraction
   for slotting into real pipelines.

In [ ]:
import numpy as np
import pandas as pd
from tcr_cliff.data import load_toy
from tcr_cliff.data.schema import AA_ALPHABET
from tcr_cliff.config import Config, EmbeddingConfig, ModelConfig
from tcr_cliff.models import train_model

df = load_toy()

cfg = Config(
    seed=0,
    embedding=EmbeddingConfig(backend='fallback', fallback_dim=64, cache_dir=None),
    model=ModelConfig(kind='baseline_lgbm'),
)
cfg.model.baseline.n_estimators = 50
model = train_model(cfg, df)

## Pick a target context (CDR3b + MHC) and a candidate target peptide

We hold the TCR (CDR3b) and MHC fixed - the therapeutic's TCR arm - and study which
peptides it might engage.

In [ ]:
# Use a real binder from the toy set as the on-target context.
anchor = df[df['binder'] == 1].iloc[0]
target_peptide = anchor['peptide']
context_cdr3b = anchor['cdr3b']
context_mhc = anchor['mhc']
context_mhc_pseudo = anchor['mhc_pseudo']
print('TCR (CDR3b) :', context_cdr3b)
print('MHC         :', context_mhc)
print('target pep  :', target_peptide)

## Enumerate all single-residue neighbours of the target peptide

For a peptide of length L over 20 amino acids there are `L * 19` substitution
neighbours (one edit, same length). Each shares the same fixed TCR/MHC context.

In [ ]:
def single_residue_neighbours(peptide):
    """Yield every length-preserving 1-substitution neighbour of `peptide`."""
    for i, original in enumerate(peptide):
        for aa in AA_ALPHABET:
            if aa != original:
                yield peptide[:i] + aa + peptide[i + 1:], i + 1, original, aa

rows = []
for neighbour, pos, orig, new in single_residue_neighbours(target_peptide):
    rows.append(
        {
            'pair_id': f'cand_{pos:02d}_{new}',
            'cdr3b': context_cdr3b,
            'peptide': neighbour,
            'mhc': context_mhc,
            'mhc_pseudo': context_mhc_pseudo,
            'binder': 0,            # unknown / placeholder for scoring
            'position': pos,
            'from_aa': orig,
            'to_aa': new,
        }
    )
cand = pd.DataFrame(rows)
print(f'enumerated {len(cand)} single-residue neighbours of {target_peptide}')
cand.head()

## Score every neighbour and flag cross-reactivity risk

In [ ]:
cand['score'] = model.predict_proba(cand)

RISK_THRESHOLD = 0.5
cand['cross_reactive_risk'] = cand['score'] >= RISK_THRESHOLD

hits = cand.sort_values('score', ascending=False)
print(f'{int(cand["cross_reactive_risk"].sum())} / {len(cand)} neighbours flagged '
      f'as potential off-target binders (score >= {RISK_THRESHOLD})')
hits[['pair_id', 'peptide', 'position', 'from_aa', 'to_aa', 'score']].head(10)

### Which positions are most permissive?
A position whose substitutions mostly keep predicted binding is a *liability
hotspot* - many self-peptides could cross-react there.

In [ ]:
by_pos = (
    cand.groupby('position')['score']
    .agg(['mean', 'max', 'count'])
    .rename(columns={'mean': 'mean_score', 'max': 'max_score'})
)
by_pos['n_risk'] = cand.groupby('position')['cross_reactive_risk'].sum()
print('per-position cross-reactivity profile (target peptide:', target_peptide, ')')
by_pos

## BONUS 1: export hits to NetTCR-2.0 format

`to_nettcr_format` maps the canonical schema to NetTCR-2.0 input columns so flagged
candidates can be re-scored by an orthogonal predictor.

In [ ]:
try:
    from tcr_cliff.data.download import to_nettcr_format
    risky = cand[cand['cross_reactive_risk']].copy()
    nettcr_df = to_nettcr_format(risky if len(risky) else cand.head())
    print('NetTCR-2.0 columns:', list(nettcr_df.columns))
    display_df = nettcr_df.head()
    # Optionally write to disk for the NetTCR pipeline:
    # from tcr_cliff.data.download import write_nettcr_csv
    # write_nettcr_csv(risky, 'crossreactivity_candidates_nettcr.csv')
    display_df
except Exception as exc:  # pragma: no cover - interop helper optional
    print('to_nettcr_format unavailable:', exc)

## BONUS 2: ANARCI / IMGT CDR3 extraction

Real pipelines start from full TCR-beta variable-domain sequences, not pre-trimmed
CDR3s. `cdr3_from_anarci` lazily imports **ANARCI**, runs **IMGT** numbering, and
extracts the CDR3 loop. ANARCI is an optional dependency, so we guard the call.

In [ ]:
try:
    from tcr_cliff.data.download import cdr3_from_anarci
    # Example full variable-domain sequence (replace with your antibody/TCR seq):
    full_vdomain = (
        'DADVTQTPRNRITKTGKRIMLECSQTKGHDRMYWYRQDPGLGLRLIYYSFDVKDINKGEISDGYS'
        'VSRQAQAKFSLSLESAIPNQTALYFCATSDRDRGYTFGSGTRLTVV'
    )
    cdr3 = cdr3_from_anarci(full_vdomain, scheme='imgt')
    print('extracted CDR3b:', cdr3)
except ImportError as exc:
    print('ANARCI not installed (optional):', exc)
    print('Install with:  pip install "tcr-cliff[anarci]"  (or conda install anarci)')
    print('Then full TCR-beta variable domains can be numbered (IMGT) to extract CDR3b.')
except Exception as exc:  # pragma: no cover - defensive
    print('CDR3 extraction unavailable:', exc)

## Summary

Starting from one target peptide and a fixed TCR/MHC context we (a) enumerated its
single-residue neighbourhood, (b) scored each neighbour, (c) flagged the ones
predicted to bind as **cross-reactivity liabilities**, (d) profiled which positions
are most permissive, and (e) exported the hits for an orthogonal NetTCR-2.0 check,
with ANARCI/IMGT bridging full variable domains into the schema.

Because off-target binders are, by construction, a single edit from the target with
a flipped intended outcome, this screen is precisely a **binding-cliff** search -
the phenomenon the whole package is built around. A cliff-*aware* model (notebook 04)
is the one you want driving this screen in production.